# S25 · A real A/B test, analysed end to end

A shopping app tests two checkout screens. We create the data and calculate each
conversion rate. Then we run the right test and report the p-value AND a likely range
for the increase. Finally we make a decision that a product manager can act on.

**New here? Read this once.**

- New to Python? Press play on each cell, from top to bottom, and read the note above
  each one. You will still finish the whole analysis.
- Not sure what a p-value means? Open `primers/p_values_in_plain_words.md` first.
- Already confident? Look for the cell marked **Stretch (optional)** near the end.
  There we show what goes wrong when you test many metrics at once.
- Any new word is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so this cell does nothing there.

In [ ]:
# This notebook uses numpy, matplotlib, scipy and statsmodels.
# Google Colab already ships all four, so there is nothing to install.
print("Setup complete - nothing to install.")

## The scenario

A shopping app tests a new checkout screen.

- **Variant A** (control): the current screen, with the usual UPI prompt.
- **Variant B** (new): a new one-tap UPI prompt.

Each user is sent to A or B at random, so the two groups are alike in every other way.
Our metric is **conversion**: did the user finish paying (yes or no)? We want to know
if B really increases conversion, or if the gap is only by chance.

## Step 1 - import our tools

We use `scipy.stats`, and two helpers from `statsmodels`. One helper runs the test for
two proportions. The other gives a likely range for the difference between them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# A statsmodels helper that compares two proportions (a two-proportion z-test).
from statsmodels.stats.proportion import proportions_ztest
# A statsmodels helper that gives a confidence interval for the difference.
from statsmodels.stats.proportion import confint_proportions_2indep

# Set a seed so we all get the same random numbers, and the same answers, every time.
np.random.seed(0)

print("Tools ready.")

## Decide the question BEFORE you look

Before we create any data, note the rule that makes an A/B test trustworthy. You fix
**one** main metric, and the sample size, *before* the experiment runs. Our main
metric is conversion, and only conversion.

A common mistake is to run the experiment first. Then the team searches through many
metrics for one that looks good. This turns a result that came by chance into a fake
"win". So we choose conversion now and we do not change it.

## Step 2 - create the experiment data

We treat each user like a coin toss: 1 = converted, 0 = did not convert. Variant A has
a true conversion chance of 11%. Variant B has a true chance of 12.8%. In real life we
would NOT know these true values. We set them here only to make the lesson concrete.
We use a large sample so that the real increase is clearly visible.

In [ ]:
# How many users each variant received.
users_A = 8000
users_B = 8000

# The TRUE conversion chance for each variant (unknown in real life).
true_rate_A = 0.110
true_rate_B = 0.128

# Simulate each user: 1 if they converted, 0 if not.
# np.random.binomial(1, p, size=n) tosses n coins, each with success chance p.
outcomes_A = np.random.binomial(1, true_rate_A, size=users_A)
outcomes_B = np.random.binomial(1, true_rate_B, size=users_B)

# The number who converted is the sum of the 1s.
converted_A = outcomes_A.sum()
converted_B = outcomes_B.sum()

print("Variant A:", converted_A, "conversions out of", users_A, "users")
print("Variant B:", converted_B, "conversions out of", users_B, "users")

## Step 3 - check that the split is balanced

A quick check: the two groups should be about the same size. If they are very
different, the experiment setup has a problem, even before we look at conversion.

In [ ]:
print("users in A:", users_A)
print("users in B:", users_B)
print("The two group sizes are close, so the random split looks fine.")

## Step 4 - calculate the conversion rates and the increase

The conversion rate is conversions divided by users. The **increase** is how much
higher B is than A, measured in percentage points.

In [ ]:
rate_A = converted_A / users_A
rate_B = converted_B / users_B

# The observed increase, in percentage points (multiply by 100 to read as %).
lift_points = (rate_B - rate_A) * 100

print("Variant A conversion rate:", round(rate_A * 100, 2), "%")
print("Variant B conversion rate:", round(rate_B * 100, 2), "%")
print("observed increase        :", round(lift_points, 2), "percentage points")

## Step 5 - draw the two conversion rates

A bar chart makes the gap clear. We write the exact rate on top of each bar.

In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(["Variant A", "Variant B"],
        [rate_A * 100, rate_B * 100],
        color=["#2E75B6", "#C0392B"])
plt.ylabel("conversion rate (%)")
plt.title("Conversion rate: A vs B")

# Write the exact rate on top of each bar.
plt.text(0, rate_A * 100 + 0.1, str(round(rate_A * 100, 2)) + "%", ha="center")
plt.text(1, rate_B * 100 + 0.1, str(round(rate_B * 100, 2)) + "%", ha="center")
plt.show()

## Step 6 - run the right test

Our metric is a yes/no event (converted or not), so we compare two **proportions**. We
use a two-proportion z-test. For this case it answers the same question as the
chi-square test from notebook 1.

In [ ]:
# proportions_ztest wants, for each group, the number of successes and the
# total number of trials, each given as an array.
successes = np.array([converted_A, converted_B])
totals = np.array([users_A, users_B])

z_statistic, p_value = proportions_ztest(successes, totals)

print("z-statistic:", round(z_statistic, 3))
print("p-value    :", round(p_value, 4))

# Read the p-value in plain words.
if p_value < 0.05:
    print()
    print("p is below 0.05: the gap is unlikely to be by chance.")
    print("We have evidence that B really does convert differently from A.")
else:
    print()
    print("p is 0.05 or above: the gap could well be by chance.")
    print("We do NOT have strong evidence of a real difference.")

## Step 7 - a likely range for the increase (confidence interval)

A p-value gives only a yes/no answer. It does not say HOW BIG the effect is. A
**likely range (confidence interval)** gives the increase AND its uncertainty: a range
where the true difference most likely lies. This is what you should report to the
business.

In [ ]:
# This returns the lower and upper ends of a 95% confidence interval for the
# difference in conversion rates (B minus A).
low, high = confint_proportions_2indep(
    converted_B, users_B,     # B successes and total
    converted_A, users_A,     # A successes and total
    method="wald")

# Convert from a fraction to percentage points so it reads like the increase above.
low_points = low * 100
high_points = high * 100

print("Likely range (95% confidence interval) for the increase (B - A):")
print("from", round(low_points, 2), "to", round(high_points, 2), "percentage points")

## Step 8 - read the likely range

If the whole range is above 0, we are fairly sure B is better. If the range includes
0, we cannot rule out "no difference". This always agrees with the p-value at the
same 5% level.

In [ ]:
if low_points > 0:
    print("The whole range is above 0, so we are fairly sure B is better.")
elif high_points < 0:
    print("The whole range is below 0, so B looks WORSE than A.")
else:
    print("The range includes 0, so we cannot be sure B is different from A.")

## Step 9 - more than chance vs big enough to matter

A result can be **statistically** significant (more than chance) but still too small
to be **practically** worth launching. Suppose the business rule is: launch B only if
the increase is at least 1 percentage point. We check our estimated increase against
that rule.

In [ ]:
# The smallest increase that the business thinks is worth the engineering effort.
practical_threshold_points = 1.0

print("observed increase       :", round(lift_points, 2), "percentage points")
print("business cares about >=  :", practical_threshold_points, "percentage points")
print()
if lift_points >= practical_threshold_points and low_points > 0:
    print("DECISION: launch B. The increase is more than chance AND big enough to matter.")
elif low_points > 0:
    print("DECISION: B is real but small. Check the business rule before you launch.")
else:
    print("DECISION: keep A. We do not have evidence that B is really better.")

### Stretch (optional) - the danger of testing many metrics

Skip this if you are new to statistics. This shows why we chose one metric at the
start. Suppose the change really does nothing, and a team checks twenty different
metrics, each at the 0.05 line. Each test has a 5% chance of a false alarm (Type I
error). So across twenty tests, we *expect* about one to look "significant" only by
chance. Let us see it happen.

In [ ]:
# Twenty metrics, and B truly equals A on every one of them (no real effect).
number_of_metrics = 20
false_alarms = 0

for metric_number in range(number_of_metrics):
    # Same true conversion chance for both groups: the change does nothing here.
    fake_A = np.random.binomial(1, 0.11, size=4000)
    fake_B = np.random.binomial(1, 0.11, size=4000)

    successes = np.array([fake_A.sum(), fake_B.sum()])
    totals = np.array([len(fake_A), len(fake_B)])
    z_statistic, p_value = proportions_ztest(successes, totals)

    # Count how many "significant" results we get even though nothing is real.
    if p_value < 0.05:
        false_alarms = false_alarms + 1

print("Metrics tested        :", number_of_metrics)
print("'Significant' by chance:", false_alarms)
print()
print("Nothing was really different, but some tests still said 'significant'.")
print("So fix ONE metric before the experiment, and correct for any extra metrics.")

## What you did

You analysed a full A/B test from start to end:

1. fixed one metric before looking,
2. created realistic conversion data,
3. calculated each conversion rate and the increase,
4. chose the right test (two proportions) and read the p-value,
5. reported a **likely range (confidence interval)** for the increase, and
6. kept **statistical** significance separate from **practical** importance before
   deciding.

A manager wants a one-line report: the decision, the size of the effect, and its
likely range. For example: *"Launch B: it increases conversion by about 1.4 points,
95% confidence interval 0.3 to 2.5."*

Next notebook: `03_how_many_users_do_we_need.ipynb`. There we calculate the sample
size *before* starting a test, so the test is big enough to find the effect we care
about.